In [ ]:
   !pip install transformers datasets evaluate accelerate scikit-learn

In [ ]:
from datasets import load_dataset
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(ds)
print(ds["train"][0])

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 92.2kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


In [ ]:
import pandas as pd

for split in ["train", "validation", "test"]:
    df = ds[split].to_pandas()
    print(f"--- {split} ---")
    print("Rows:", len(df))
    print("Empty texts:", (df["text"].str.strip() == "").sum())
    print("Duplicate texts:", df["text"].duplicated().sum())
    print("Class balance:\n", df["label"].value_counts())
    print("Avg words per review:", df["text"].str.split().str.len().mean().round(1))
    print()

--- train ---
Rows: 8530
Empty texts: 0
Duplicate texts: 0
Class balance:
 label
1    4265
0    4265
Name: count, dtype: int64
Avg words per review: 21.0

--- validation ---
Rows: 1066
Empty texts: 0
Duplicate texts: 0
Class balance:
 label
1    533
0    533
Name: count, dtype: int64
Avg words per review: 21.0

--- test ---
Rows: 1066
Empty texts: 0
Duplicate texts: 0
Class balance:
 label
1    533
0    533
Name: count, dtype: int64
Avg words per review: 21.2



In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def prep(b): return tok(b["text"], truncation=True, padding="max_length", max_length=128)
ds_tok = ds.map(prep, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1})

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import numpy as np, evaluate
from transformers import TrainingArguments, Trainer
acc = evaluate.load("accuracy")
def metrics(p): return acc.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

args = TrainingArguments("my-model", num_train_epochs=3, per_device_train_batch_size=16,
                         eval_strategy="epoch", save_strategy="epoch", learning_rate=2e-5,
                         report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=ds_tok["train"],
                  eval_dataset=ds_tok["validation"], compute_metrics=metrics)
trainer.train()
print(trainer.evaluate(ds_tok["test"]))

Epoch,Training Loss,Validation Loss,Accuracy
1,0.423457,0.387263,0.831144
2,0.255017,0.376139,0.854597
3,0.160400,0.494627,0.853659


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.160400,0.562337,3,0.835835


{'eval_loss': 0.5623365640640259, 'eval_accuracy': 0.8358348968105066}


In [ ]:
trainer.save_model("final-model"); tok.save_pretrained("final-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final-model/tokenizer_config.json', 'final-model/tokenizer.json')

In [ ]:
   from huggingface_hub import login
   login()

In [ ]:
model.push_to_hub("aqsaahsan/sentiment-distilbert")
tok.push_to_hub("aqsaahsan/sentiment-distilbert")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...sshfmfx/model.safetensors:   2%|2         | 5.65MB /  268MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/aqsaahsan/sentiment-distilbert/commit/101151754fcc5ba1cdc4085994782f9c80e01889', commit_message='Upload tokenizer', commit_description='', oid='101151754fcc5ba1cdc4085994782f9c80e01889', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aqsaahsan/sentiment-distilbert', endpoint='https://huggingface.co', repo_type='model', repo_id='aqsaahsan/sentiment-distilbert'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model="aqsaahsan/sentiment-distilbert")
print(clf("This movie was fantastic!"))
print(clf("Worst film I have ever seen."))

config.json:   0%|          | 0.00/784 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9950553178787231}]
[{'label': 'NEGATIVE', 'score': 0.9926515221595764}]
